# AgriRAG — Part B: Groq + Supabase + TF-IDF RAG

This notebook documents the retrieval pipeline and eight test scenarios. The production app calls Groq for non-dosage answers; dosage is gated in deterministic application code before any model call.

## System prompt — Version 1

You are a farmer assistant. Answer crop questions politely and cite sources. Do not invent pesticide dosage.

## System prompt — Version 2 (refined)

You are AgriRAG, a cautious farmer-support assistant. Use only the supplied retrieved extension notes and profile/weather context. Reply in the selected language. Cite each recommendation as [Guide title — Section]. Never invent pesticide names, doses, intervals, pre-harvest intervals or diagnoses. Dosage requests must be blocked unless deterministic code verifies an exact crop+pest+approved-guide match. If evidence is missing, symptoms are ambiguous, retrieval is weak or advice is region-specific, refuse to guess and escalate to Agriculture Officer/KVK. Label synthetic weather as test data, never as a forecast. Do not claim to be an official service.

### What improved

Version 2 adds explicit retrieval-only grounding, exact citation format, language instruction, strict dosage gating, uncertain-pest escalation and synthetic-weather labelling.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'rag.py').exists():
    ROOT = Path(__file__).resolve().parent if '__file__' in dir() else ROOT
sys.path.insert(0, str(ROOT))
from rag import retrieve_top_chunks, safe_dosage_match, is_dosage_request, build_grounded_prompt
from local_data import load_guides_csv

chunks = load_guides_csv()
p = {'district': 'Warangal', 'crop': 'Tomato', 'season': 'Rabi', 'soil': 'Sandy loam', 'land_acres': 1, 'language': 'English', 'name': 'Demo'}
retrieved = retrieve_top_chunks('What soil is suitable for tomato?', p, chunks, 3)
print('Top-3 retrieval demo:')
[(x['guide_title'], x['section'], round(x['_score'], 3), x['source_url']) for x in retrieved]

## Eight required assignment tests

Use the deployed app or `test_pipeline.py` for executable retrieval/safety checks. The following test matrix records expected behaviour; run the application against the seeded Supabase project and capture actual outputs before submission.

In [ ]:
test_matrix=[('T1','Paddy irrigation','Cited paddy water guidance'),('T2','Cotton pest scouting','Cited IPM guidance'),('T3','Tomato soil','Cited soil guidance'),('T4','Paddy nutrients','Cited nutrient guidance'),('T5','Telugu/Hindi question','Selected language answer'),('T6','Covered tomato NSKE request','Exact guide dosage with citation'),('T7','Uncovered cotton dosage','Refusal + officer escalation'),('T8','Unknown pest','Low-confidence refusal + escalation')]
import pandas as pd
pd.DataFrame(test_matrix,columns=['Test','Input','Expected result'])

## Before/after refinement

1. Before: a broad dosage question could retrieve a dosage note from an unrelated crop. After: the code requires tomato + serpentine leaf miner + NSKE and an approved matching record before revealing the single documented demonstration dosage.
2. Before: TF-IDF overlap could be mistaken for confirmed diagnosis. After: pest results are explicitly labelled possible matches; weak overlap triggers escalation.
3. Before: synthetic weather could be mistaken for real forecast data. After: the UI and prompt explicitly identify it as synthetic.

## Verification table

| Suggestion | How checked | Result | Change |
|---|---|---|---|
| Top-3 retrieval retains citation metadata | Inspect retrieved chunk dictionaries | Title, section and URL retained | Citation fields passed into Groq prompt |
| Cotton/paddy/tomato advice grounded | Compare app responses with source guide pages | Verify before submission | Remove unsupported claims |
| Dosage only for exact match | Test covered tomato NSKE vs cotton dosage | Covered request only passes gate | Deterministic dosage gate |
| Unknown pest escalates | Enter ambiguous symptoms | Low-confidence response | No diagnosis or chemical dose |
| Supabase data persistence | Save profile and chat; inspect tables | Verify in Supabase Table Editor | Keep service role key out of app |
| Weather not treated as forecast | Inspect app labels and weather CSV | Synthetic label visible | Add warning in UI and prompt |

## AI chatbot log

Cursor Agent (Composer) implemented the Streamlit UI, Groq integration, Supabase schema/CRUD, TF-IDF retrieval, dosage gate, pest matcher, CSV knowledge files, tests and documentation. See `CHATBOT_LOG.md` for the numbered assistance log. Evidence to retain: terminal test output, Supabase table rows, app screenshots and corrections after running. Do not claim live Groq/Supabase tests passed unless you actually ran them with credentials.

## Reflection (about 180 words)

This project uses retrieval and an LLM together, but the model is not treated as an agricultural authority. A first risk is retrieval mismatch: TF-IDF ranks word overlap and can select a similar-sounding note that belongs to another crop or pest. I addressed this by including crop and profile terms in the query, retaining guide title and section metadata, and reviewing the top-three chunks during testing. A second risk is unsafe dosage generation. A language model may produce a plausible but unsupported amount, so dosage requests are intercepted in deterministic code before the Groq call. Only an exact approved crop-and-pest guide match can pass; all other dosage requests are refused and referred to the officer. A third limitation is that symptom keywords cannot confirm a pest and synthetic weather is not a real forecast. The interface labels pest results as possible matches and clearly marks weather as synthetic. I would refer a farmer to the Agriculture Officer/KVK when the pest is uncertain, crop damage is severe, symptoms may indicate disease, a chemical dose or pre-harvest interval is requested, or local label and weather conditions must be checked. The app supports extension work but does not replace field inspection.